In [1]:
!pip install -q polars faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 82.4 MB/s eta 0:00:00


In [2]:
import os
import gc
import shutil
import pandas as pd
import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from google.colab import drive
import traceback

# 1. Mount Drive (Chỉ gọi 1 lần duy nhất)
drive.mount('/content/drive')

# 2. Tối ưu phân mảnh VRAM
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 3. Seed everything an toàn (Bao quát Multi-GPU)
def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng thiết bị: {device}")

Mounted at /content/drive
Đang sử dụng thiết bị: cuda


In [4]:
# ---------------------------------------------------------
# CẤU HÌNH ĐƯỜNG DẪN
# ---------------------------------------------------------
DRIVE_TRAIN_PATH = '/content/drive/MyDrive/AmazonDataset/train_interactions.parquet'
WORKING_DIR      = '/content'

# Copy từ Drive sang Local Colab để tăng tốc I/O
LOCAL_TRAIN_PATH = os.path.join(WORKING_DIR, 'train_interactions.parquet')
print("Đang copy dữ liệu từ Drive vào Local Disk của Colab...")
shutil.copy2(DRIVE_TRAIN_PATH, LOCAL_TRAIN_PATH)

MODEL_SAVE_PATH  = os.path.join(WORKING_DIR, 'sasrec_model.pth')
SASREC_CAND_PATH = os.path.join(WORKING_DIR, 'sasrec_candidates.parquet')
CHUNK_DIR        = os.path.join(WORKING_DIR, 'sasrec_chunks')

EMBED_DIM  = 64
MAX_LEN    = 50
BATCH_SIZE = 4096
EPOCHS     = 20

print("Đang tiền xử lý chuỗi bằng Polars...")
# BƯỚC KHẮC PHỤC CHÍNH: Thêm .unique() để dọn sạch rác nhân bản 100%
df_train_pl = pl.read_parquet(LOCAL_TRAIN_PATH, columns=['mapped_user_id', 'mapped_item_id', 'timestamp']).unique()

num_users = df_train_pl['mapped_user_id'].max() + 1
num_items = df_train_pl['mapped_item_id'].max() + 1

user_seqs_pl = (
    df_train_pl.sort(['mapped_user_id', 'timestamp'])
    .group_by('mapped_user_id')
    .agg(pl.col('mapped_item_id'))
)

mapped_user_ids = user_seqs_pl['mapped_user_id'].to_numpy()
item_lists = user_seqs_pl['mapped_item_id'].to_list()

X_sas_train = np.zeros((len(item_lists), MAX_LEN), dtype=np.int32)
for idx, seq in enumerate(item_lists):
    s = seq[-MAX_LEN:]
    X_sas_train[idx, MAX_LEN-len(s):] = s

del df_train_pl, user_seqs_pl, item_lists
gc.collect()

print(f"Tổng số chuỗi (Users): {len(X_sas_train):,}")
print(f"Kích thước từ điển Items (kể cả padding 0): {num_items:,}")

Đang copy dữ liệu từ Drive vào Local Disk của Colab...
Đang tiền xử lý chuỗi bằng Polars...
Tổng số chuỗi (Users): 2,181,749
Kích thước từ điển Items (kể cả padding 0): 626,748


In [5]:
class SASRec(nn.Module):
    def __init__(self, n_items, embed_dim, max_len):
        super().__init__()
        self.item_emb = nn.Embedding(n_items, embed_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, embed_dim)

        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=1,
            batch_first=True,
            dim_feedforward=embed_dim*2,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=1)

    def forward(self, seqs):
        pos = torch.arange(seqs.size(1), device=seqs.device).unsqueeze(0).expand_as(seqs)
        mask = (seqs == 0)
        out = self.transformer(self.item_emb(seqs) + self.pos_emb(pos), src_key_padding_mask=mask)
        return out[:, -1, :]

model_sasrec = SASRec(num_items, EMBED_DIM, MAX_LEN).to(device)

if torch.cuda.device_count() > 1:
    print(f"Sử dụng {torch.cuda.device_count()} GPUs bằng DataParallel!")
    model_sasrec = nn.DataParallel(model_sasrec)

optimizer = torch.optim.Adam(model_sasrec.parameters(), lr=0.001)
scaler = torch.amp.GradScaler('cuda')

print("Đang đẩy dữ liệu lên VRAM...")
X_tensor = torch.tensor(X_sas_train, dtype=torch.long, device=device)
del X_sas_train
gc.collect()

model_sasrec.train()
for ep in range(EPOCHS):
    idx_perm = torch.randperm(len(X_tensor), device=device)
    loss_ep, n_batches = 0, 0
    pbar = tqdm(range(0, len(X_tensor), BATCH_SIZE), desc=f"SASRec Epoch {ep+1}/{EPOCHS}", leave=False)

    for i in pbar:
        b_idx = idx_perm[i:i+BATCH_SIZE]
        batch_seqs = X_tensor[b_idx]

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            u_reps = model_sasrec(batch_seqs[:, :-1])

            pos_items = batch_seqs[:, -1]
            neg_items = torch.randint(1, num_items, (len(batch_seqs),), device=device)

            # Đảm bảo lấy đúng model (thường là DataParallel.module)
            base_model = model_sasrec.module if hasattr(model_sasrec, 'module') else model_sasrec

            pos_embs = base_model.item_emb(pos_items)
            neg_embs = base_model.item_emb(neg_items)

            pos_logits = (u_reps * pos_embs).sum(dim=-1)
            neg_logits = (u_reps * neg_embs).sum(dim=-1)

            labels_pos = torch.ones_like(pos_logits)
            labels_neg = torch.zeros_like(neg_logits)

            loss = F.binary_cross_entropy_with_logits(pos_logits, labels_pos) + \
                   F.binary_cross_entropy_with_logits(neg_logits, labels_neg)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_ep += loss.item()
        n_batches += 1
        pbar.set_postfix(loss=f"{loss_ep/n_batches:.4f}")

    if (ep+1) % 5 == 0:
        print(f"SASRec Epoch {ep+1:02d}/{EPOCHS} | Avg Loss: {loss_ep/max(n_batches,1):.4f}")

base_model = model_sasrec.module if hasattr(model_sasrec, 'module') else model_sasrec
torch.save(base_model.state_dict(), MODEL_SAVE_PATH)
print(f"✅ Đã lưu trọng số tại: {MODEL_SAVE_PATH}")

/tmp/ipykernel_13691/4268739385.py:14: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, num_layers=1)


Đang đẩy dữ liệu lên VRAM...


SASRec Epoch 1/20:   0%|          | 0/533 [00:00<?, ?it/s]

SASRec Epoch 2/20:   0%|          | 0/533 [00:00<?, ?it/s]

SASRec Epoch 3/20:   0%|          | 0/533 [00:00<?, ?it/s]

...

SASRec Epoch 17/20:   0%|          | 0/533 [00:00<?, ?it/s]

SASRec Epoch 18/20:   0%|          | 0/533 [00:00<?, ?it/s]

SASRec Epoch 19/20:   0%|          | 0/533 [00:00<?, ?it/s]

SASRec Epoch 20/20:   0%|          | 0/533 [00:00<?, ?it/s]

SASRec Epoch 20/20 | Avg Loss: 0.9130
✅ Đã lưu trọng số tại: /content/sasrec_model.pth


In [6]:
torch.cuda.empty_cache()
gc.collect()

model_sasrec.eval()
base_model_sas = model_sasrec.module if hasattr(model_sasrec, 'module') else model_sasrec

os.makedirs(CHUNK_DIR, exist_ok=True)
INFER_BATCH_SIZE = 512
CHUNK_SIZE = 1000
K_RANK = 100 # Sinh top 100

print(f"Đang trích xuất Top {K_RANK} (Batch size: {INFER_BATCH_SIZE})...")

users_list = []
items_list = []
chunk_idx = 0

with torch.no_grad():
    # Giữ nguyên toàn bộ ma trận (kể cả index 0)
    i_embs = torch.nn.functional.normalize(base_model_sas.item_emb.weight, p=2, dim=1)

    pbar = tqdm(range(0, len(X_tensor), INFER_BATCH_SIZE), desc="Inference")
    for i in pbar:
        batch_seqs = X_tensor[i:i+INFER_BATCH_SIZE]
        u_batch_ids = mapped_user_ids[i:i+INFER_BATCH_SIZE]

        with torch.amp.autocast('cuda'):
            u_reps = base_model_sas(batch_seqs)
            u_reps = torch.nn.functional.normalize(u_reps, p=2, dim=1)

            scores = torch.matmul(u_reps, i_embs.T)
            # CỰC KỲ QUAN TRỌNG: Che giấu item 0 (Padding)
            scores[:, 0] = float('-inf')

            _, top_idx = torch.topk(scores, K_RANK, dim=1)

        users_list.append(np.repeat(u_batch_ids, K_RANK).astype(np.int32))
        items_list.append(top_idx.cpu().numpy().flatten().astype(np.int32))

        del scores, u_reps, top_idx

        # Ghi chunk
        if len(users_list) >= CHUNK_SIZE or (i + INFER_BATCH_SIZE) >= len(X_tensor):
            final_u = np.concatenate(users_list)
            final_i = np.concatenate(items_list)

            df_chunk = pd.DataFrame({
                'mapped_user_id': final_u,
                'mapped_item_id': final_i
            })

            # Tính rank siêu an toàn (K_RANK = 100)
            df_chunk['sasrec_rank'] = np.tile(np.arange(1, K_RANK+1, dtype=np.int8), len(final_u) // K_RANK)

            chunk_path = f"{CHUNK_DIR}/chunk_{chunk_idx}.parquet"
            df_chunk.to_parquet(chunk_path)

            users_list, items_list = [], []
            chunk_idx += 1
            gc.collect()

del X_tensor, i_embs
torch.cuda.empty_cache()
gc.collect()

# Gộp file an toàn
try:
    print("Đang gộp các file chunk bằng Polars...")
    lf_sasrec = pl.scan_parquet(f'{CHUNK_DIR}/chunk_*.parquet')
    lf_sasrec.sink_parquet(SASREC_CAND_PATH)

    shutil.rmtree(CHUNK_DIR)
    print(f"✅ Gộp thành công. Dọn rác xong. Kết quả Local: {SASREC_CAND_PATH}")

    # Copy sang Google Drive
    DRIVE_OUTPUT_PATH = '/content/drive/MyDrive/AmazonDataset/sasrec_candidates.parquet'
    os.makedirs(os.path.dirname(DRIVE_OUTPUT_PATH), exist_ok=True)
    shutil.copy2(SASREC_CAND_PATH, DRIVE_OUTPUT_PATH)
    print(f"✅ Đã copy an toàn sang Google Drive: {DRIVE_OUTPUT_PATH}")

    # Dọn dẹp file train ban đầu
    if os.path.exists(LOCAL_TRAIN_PATH):
        os.remove(LOCAL_TRAIN_PATH)
        print("Đã dọn dẹp file Train trên Local Disk!")

except Exception as e:
    print("❌ LỖI TRONG QUÁ TRÌNH GỘP PARQUET!")
    print(traceback.format_exc())
    print("Các file chunk tạm vẫn được giữ lại tại:", CHUNK_DIR)

Đang trích xuất Top 100 (Batch size: 512)...


Inference:   0%|          | 0/4262 [00:00<?, ?it/s]

Đang gộp các file chunk bằng Polars...
✅ Gộp thành công. Dọn rác xong. Kết quả Local: /content/sasrec_candidates.parquet
✅ Đã copy an toàn sang Google Drive: /content/drive/MyDrive/AmazonDataset/sasrec_candidates.parquet
Đã dọn dẹp file Train trên Local Disk!
